# ProtSpace — Embedding Annotation Transfer (EAT)

This notebook demonstrates **Embedding Annotation Transfer (EAT)** with `protspace transfer`.
For each query protein that lacks an annotation value, the command finds the closest
annotated reference protein in pLM embedding space and transfers its label, together
with a reliability index in [0, 1]. The exact confidence formula depends on `--metric` and `--k`:

- Default (`--metric cosine`, `--k 1`): `confidence = clamp(1 - cosine_distance, 0, 1)` (cosine distance in [0, 2]). Cosine is the default because the confidence is naturally bounded and directly interpretable.
- `--metric euclidean` (`--k 1`): `confidence = 0.5 / (0.5 + distance)` — the published goPredSim transform, calibrated for ProtT5; on embedding spaces with larger raw distances treat it as a ranking rather than a calibrated probability.
- `--k > 1`: the goPredSim mean reliability — `(1/m) * sum` of the per-neighbour similarity over the `k` nearest neighbours carrying the chosen label, where `m = min(k, number of references)`. Because of this normalization, values are not comparable across different `--k` settings.

The method follows the goPredSim approach introduced in:

- Littmann et al., *Sci Rep* 2021 — [DOI 10.1038/s41598-020-80786-0](https://doi.org/10.1038/s41598-020-80786-0)
- Heinzinger et al., *NAR Genom Bioinform* 2022 — [DOI 10.1093/nargab/lqac043](https://doi.org/10.1093/nargab/lqac043)

Distances are computed in the original high-dimensional embedding space (HDF5),
not in any 2-D/3-D projection. The curated source column is left untouched;
results are written as `COL__pred_value`, `COL__pred_confidence`, and
`COL__pred_source` columns in the bundle's annotations table. `COL__pred_source`
is the reference protein the label was transferred from — provenance for a
connector line or "transferred from &lt;neighbour&gt;" tooltip in the web frontend.

📚 [GitHub](https://github.com/tsenoner/protspace) · [CLI Reference](https://protspace.app/docs/guide/python-cli#protspace-transfer) · [Annotation Reference](https://protspace.app/docs/explore/eat)

In [ ]:
# @title 1. Install {display-mode: "form"}
# Installed via subprocess, not `!pip`: `%%capture` only works on line 1, which
# `# @title` must occupy for Colab to render the form header, and `-qqq` alone
# does not silence Colab's download bars or pip's dependency-conflict block.
# capture_output hides both; the output is shown only if the install fails.
import subprocess
import sys

_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-qqq", "--disable-pip-version-check", "protspace"],
    capture_output=True,
    text=True,
)
if _pip.returncode:
    raise SystemExit(_pip.stdout[-3000:] + _pip.stderr[-3000:])


In [ ]:
# @title 2. Fetch example data {display-mode: "form"}
# @markdown Fetches the **Snake Toxin** example into this Colab runtime: 5,015 proteins
# @markdown (1,997 reviewed Swiss-Prot + 3,018 unreviewed TrEMBL) and their ProtT5
# @markdown embeddings. 849 of them have no `protein_families` value, which is what EAT
# @markdown fills in below. Swap these two files for your own bundle and `.h5` to run on
# @markdown your own data.
import shutil
import urllib.request
from pathlib import Path

_BASE = "https://github.com/tsenoner/protspace/releases/download/examples/"


def _fetch(name, dest):
    """Download once, then reuse. Interrupted fetches are retried, not cached.

    urlretrieve streams straight into its destination, so an aborted transfer
    leaves a truncated file that still satisfies exists() and would be reused
    forever. Landing on a .part file and renaming only once complete makes the
    destination appear only when it is whole.
    """
    if Path(dest).exists():
        print(f"Reusing {dest} already in this Colab runtime.")
        return
    print(f"Fetching {dest} into this Colab runtime...")
    _part = Path(str(dest) + ".part")
    urllib.request.urlretrieve(_BASE + name, _part)
    _part.replace(dest)


# Both downloads are cached under their own names, then the bundle is copied to
# the working path: step 3 writes its results back over that copy, so every run
# has to start from a pristine one. Copying 320 KB locally beats re-downloading
# it, and it lets a re-run work with no network once the first one succeeded.
_fetch("snake_toxin.parquetbundle", "snake_toxin.parquetbundle")
_fetch("snake_toxin_prot_t5.h5", "embeddings.h5")
shutil.copyfile("snake_toxin.parquetbundle", "results.parquetbundle")
print("Done.")


In [ ]:
# @title 3. Run transfer {display-mode: "form"}
# @markdown With no `--query-*` / `--reference-*` filters, transfer runs **within the
# @markdown bundle**: every protein missing a value in the `-t` column is a query, every
# @markdown protein holding one is a reference. Here that would be all 849 gaps.
# @markdown
# @markdown This example narrows it on purpose, to what the filters are for: queries are the
# @markdown **unreviewed TrEMBL** entries, references only the **reviewed Swiss-Prot** ones,
# @markdown so no label is copied from an uncurated source. Passing just one side is fine;
# @markdown the other stays unrestricted. `--query-id-prefix` / `--reference-id-prefix`
# @markdown filter by ID prefix instead.
# @markdown
# @markdown The curated column is never overwritten -- results land in
# @markdown `protein_families__pred_*`.
!protspace transfer \
  -b results.parquetbundle \
  -e embeddings.h5:prot_t5 \
  -t protein_families \
  -o results.parquetbundle \
  --query-where 'reviewed~TrEMBL' \
  --reference-where 'reviewed~Swiss-Prot'


In [ ]:
# @title 4. Read predictions back {display-mode: "form"}
# @markdown Loads the updated bundle and shows the curated column beside its
# @markdown predicted value, confidence and source neighbour. Expect ~838 filled rows.
import io
import pyarrow.parquet as pq
from protspace.data.io.bundle import read_bundle

parts, _ = read_bundle("results.parquetbundle")
df = pq.read_table(io.BytesIO(parts[0])).to_pandas()

# Show the curated column next to its predicted value / confidence / source
# overlay for the proteins that received a transferred label.
pred_value_cols = [c for c in df.columns if c.endswith("__pred_value")]
if pred_value_cols:
    col = pred_value_cols[0][: -len("__pred_value")]
    overlay = [
        c
        for c in (col, f"{col}__pred_value", f"{col}__pred_confidence", f"{col}__pred_source")
        if c in df.columns
    ]
    display(df.loc[df[f"{col}__pred_value"].notna(), overlay].head())
else:
    display(df.head())